# 04. Retrieval Documents, Facets, Graph, and Temporal Indices — Herbal Supplements

This notebook materializes the fixed item-side artifacts consumed by later retrieval and reranking stages. It loads the canonical item schema from Notebook 02, removes discontinued products, and retains 27,253 active parent items.

The dense and sparse item documents combine catalog metadata, Brand, normalized functional facets, and frozen review-derived item signals. Their provenance remains separated in the exported schema. The long-form facet table and item–entity graph distinguish catalog-functional facets, Brand, and review-derived rows while excluding generic category anchors, generic utility terms, context-dependent utility terms, and diagnostic fields from the production graph.

Brand is retained as a separate preference facet in item text, graph, candidate, and profile representations. It is not a product-functional facet and remains unavailable to synthetic-query construction.

The temporal artifacts contain item- and facet-level review timestamps and counts for later point-in-time activity features. They contain no raw review language, ratings, sentiment, target-review text, or user history. Downstream use must enforce `review_timestamp_ms < target_timestamp_ms`.

The exports include item documents, item facets, graph edges and nodes, facet vocabularies, temporal review indices, coverage audits, shared contracts, and provenance manifests. No query is issued, retrieval method is selected, candidate pool is generated, or ranker is fitted here.


## Artifact Scope

This is an item-side construction notebook rather than a retrieval experiment. It freezes the documents and structured artifacts used by later notebooks under one evidence contract. Retrieval comparisons, candidate generation, and model fitting occur downstream.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ==== Load Libraries ====
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

In [3]:
# ==== Define Inputs, Outputs, and Evidence Masks ====
CATEGORY_ID = "herbal"
CATEGORY_FOLDER = "herbal_supplements"
CATEGORY_LABEL = "Herbal Supplements"
STAGE = "stage0_retrieval_artifact"

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_ITEMS_DIR = PROJECT_ROOT / "data" / "processed" / "items"
OUTPUT_DIR = PROCESSED_ITEMS_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_PATH = OUTPUT_DIR / "herbal_item_schema.parquet"
REVIEWS_PATH = RAW_DIR / "reviews_Herbal_Supplements_W2_2019_2022.parquet"
ACTIVE_SCHEMA_PATH = OUTPUT_DIR / "items_Herbal_Supplements_W2_2019_2022_schema_active.parquet"

ITEM_DOCS_DENSE_PATH = OUTPUT_DIR / "item_docs_dense_herbal.parquet"
ITEM_DOCS_SPARSE_PATH = OUTPUT_DIR / "item_docs_sparse_herbal.parquet"
ITEM_DOCS_PATH = OUTPUT_DIR / "item_docs_herbal.parquet"
ITEMS_FACETS_PATH = OUTPUT_DIR / "items_facets_herbal.parquet"
ITEM_GRAPH_LONG_PATH = OUTPUT_DIR / "item_graph_long_herbal.parquet"
ITEM_KG_EDGES_PATH = OUTPUT_DIR / "item_kg_edges_herbal.parquet"
GRAPH_EDGES_PATH = OUTPUT_DIR / "graph_edges_herbal.parquet"
PRODUCT_NODES_PATH = OUTPUT_DIR / "product_nodes_herbal.parquet"
ENTITY_NODES_PATH = OUTPUT_DIR / "entity_nodes_herbal.parquet"
FACET_VOCAB_PATH = OUTPUT_DIR / "facet_vocab_herbal.json"
FACET_VOCAB_TABLE_PATH = OUTPUT_DIR / "facet_vocab_herbal.parquet"
FACET_COVERAGE_PATH = OUTPUT_DIR / "items_facets_coverage_herbal.csv"
DOC_COVERAGE_PATH = OUTPUT_DIR / "item_docs_coverage_herbal.csv"
CORE_FACET_FILTER_AUDIT_PATH = OUTPUT_DIR / "herbal_core_facet_filter_audit.csv"
MANIFEST_PATH = OUTPUT_DIR / "retrieval_artifact_manifest_herbal.json"

ITEM_REVIEW_TIME_INDEX_PATH = OUTPUT_DIR / "item_review_time_index_herbal.parquet"
ITEM_REVIEW_DAILY_COUNTS_PATH = OUTPUT_DIR / "item_review_daily_counts_herbal.parquet"
ENTITY_REVIEW_DAILY_COUNTS_PATH = OUTPUT_DIR / "entity_review_daily_counts_herbal.parquet"
ITEM_TEMPORAL_SUMMARY_PATH = OUTPUT_DIR / "item_temporal_summary_herbal.parquet"
TEMPORAL_ARTIFACT_MANIFEST_PATH = OUTPUT_DIR / "temporal_artifact_manifest_herbal.json"

ACTIVE_SCHEMA_COMPATIBILITY_PATHS = [
    OUTPUT_DIR / "herbal_item_schema_full.parquet",
]
ITEM_DOCS_COMPATIBILITY_PATHS = [
    OUTPUT_DIR / "herbal_item_docs.parquet",
]
ITEMS_FACETS_COMPATIBILITY_PATHS = [
    OUTPUT_DIR / "herbal_items_facets.parquet",
]

CATEGORY_REQUIRED_FACET_COLS = [
    "facet_category_text",
    "facet_form_text",
    "facet_ingredient_text",
    "facet_benefit_text",
    "facet_claim_text",
    "facet_flavor_text",
    "facet_audience_text",
]
CATEGORY_OPTIONAL_FACET_SPECS = [
    ("facet_flavor_text", "flavor", "HAS_FLAVOR"),
    ("facet_audience_text", "audience", "HAS_AUDIENCE"),
]

TEMPORAL_FACET_TYPES = [
    "brand",
    "category",
    "form",
    "ingredient",
    "benefit",
    "claim",
    "flavor",
    "audience",
]
TEMPORAL_WINDOWS_DAYS = [30, 90, 180]
TEMPORAL_REVIEW_TIMESTAMP_COL = "review_timestamp_ms"
TEMPORAL_REVIEW_STRICT_RULE = "review_timestamp_ms < target_timestamp_ms"

PRODUCTION_EVIDENCE_SCOPE = "catalog_metadata_functional_facets_and_historical_review_signals"
PRODUCTION_FACET_EXPORT_OPTION = "B_all_rows_with_reliable_flags"
CORE_GRAPH_MASK_DESCRIPTION = (
    "is_product_functional_facet == True and is_query_safe == True and "
    "is_brand == False and is_review_derived == False and "
    "is_generic_category_anchor == False and is_generic_utility_token == False and "
    "is_context_dependent_utility_token == False and is_metadata_facet_source == True and "
    "is_disallowed_nonfacet_source == False"
)
BRAND_GRAPH_MASK_DESCRIPTION = (
    "facet_role == 'brand' and is_brand == True and is_review_derived == False and "
    "is_metadata_facet_source == True and is_disallowed_nonfacet_source == False"
)
GLOBAL_REVIEW_GRAPH_MASK_DESCRIPTION = (
    "is_core_graph_facet == True OR is_brand_graph_facet == True OR "
    "(is_review_derived == True and is_brand == False and "
    "is_generic_category_anchor == False and is_generic_utility_token == False and "
    "is_context_dependent_utility_token == False and is_disallowed_nonfacet_source == False)"
)


FORBIDDEN_FACET_SOURCE_TERMS = ("identifier", "count", "policy", "diagnostic")

EXPECTED_FACET_POLICY_VERSION = "harmonized_v2_global_review_brand_retrieval_profile"
EXPECTED_BRAND_POLICY = "separate_preference_facet__query_unsafe__retrieval_profile_graph_safe"
EXPECTED_IDENTIFIER_POLICY = "diagnostic_only__excluded_from_query_safe_profile_and_canonical_retrieval_text"

print("Input:", SCHEMA_PATH)
print("Input:", REVIEWS_PATH)
print("Output:", ITEM_DOCS_PATH)


Input: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_item_schema.parquet
Input: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/raw/reviews_Herbal_Supplements_W2_2019_2022.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/item_docs_herbal.parquet


### Merged Item-Evidence Contract

Production item documents combine catalog metadata and frozen review-derived item signals. The facet export retains source and safety flags for every row, while the production graph admits only validated catalog-functional facets, the separate Brand channel, and eligible review-derived signal rows. Raw review text and user-specific history are excluded.


### Brand Treatment

Brand is retained in retrieval text, graph, candidate, and user-profile representations as a separate preference facet. It is excluded from synthetic-query evidence and from product-functional facet metrics.


In [4]:
# ==== Declare the Shared Retrieval-Artifact Contract ====
COMMON_RETRIEVAL_ARTIFACT_FRAMEWORK_VERSION = "common_retrieval_artifact_framework_v5_global_review_brand_profile"

COMMON_FACET_TYPE_TO_ROLE = {
    "brand": "brand",
    "category": "category_or_product_type",
    "form": "form_texture",
    "audience": "target_context",
    "flavor": "sensory",
    "ingredient": "ingredient_or_composition",
    "benefit": "need_benefit_concern",
    "claim": "claim_constraint",
    "claim_diet": "claim_constraint",
    "review_reputation_benefit": "review_derived_signal",
    "review_reputation_ingredient": "review_derived_signal",
    "review_reputation_form": "review_derived_signal",
    "review_reputation_claim_diet": "review_derived_signal",
    "review_reputation_flavor": "review_derived_signal",
    "review_reputation": "review_derived_signal",
}
COMMON_GENERIC_CATEGORY_ANCHORS = {
    "herbal", "supplement", "dietary supplement", "herbal supplement"
}
COMMON_GENERIC_UTILITY_TOKENS = {
    "support", "supports", "help", "helps", "promote", "promotes", "boost",
    "formula", "blend", "complex", "product", "solution",
}
COMMON_CONTEXT_DEPENDENT_UTILITY_TOKENS = {
    "daily", "natural", "wellness", "health", "care", "routine",
}
COMMON_SPECIFIC_PHRASE_EXCEPTIONS = {
    "immune support", "digestive support", "sleep support", "joint support",
    "stress support", "energy support", "liver support",
}
HERBAL_MULTIWORD_ENTITY_FRAGMENT_TOKENS = {
    "milk", "olive", "saw", "seed", "dong", "john", "biloba", "holy",
    "evening", "horny", "vera", "red", "black", "fruit",
}
COMMON_FUNCTIONAL_FACET_ROLES = {
    "category_or_product_type",
    "form_texture",
    "ingredient_or_composition",
    "need_benefit_concern",
    "claim_constraint",
    "target_context",
    "sensory",
}
COMMON_BRAND_POLICY = (
    "Brand is retained as a separate query-unsafe preference facet and is active in "
    "retrieval, graph, candidate, and user-profile representations."
)
COMMON_FUNCTIONAL_FACET_DEFINITION = (
    "Metadata product-functional facets excluding brand, review-derived rows, generic category anchors, "
    "standalone generic utility tokens, standalone context-dependent utility tokens, and diagnostic/count fields."
)
COMMON_BRAND_DIAGNOSTIC_METRICS = [
    "brand_overlap", "brand_match", "seen_brand_match", "profile_brand_overlap",
    "weighted_facet_overlap_with_brand", "weighted_facet_overlap_no_brand", "brand_share_of_overlap",
]

COMMON_RETRIEVAL_ARTIFACT_CONTRACT_PATH = OUTPUT_DIR / "herbal_retrieval_artifact_common_contract.json"
COMMON_FACET_ROLE_COVERAGE_PATH = OUTPUT_DIR / "herbal_facet_role_coverage.csv"
COMMON_FLAG_SELF_CHECK_PATH = OUTPUT_DIR / "common_framework_flag_self_check_herbal.csv"

COMMON_RETRIEVAL_ARTIFACT_CONTRACT = {
    "category_id": CATEGORY_ID,
    "framework_version": COMMON_RETRIEVAL_ARTIFACT_FRAMEWORK_VERSION,
    "evidence_scope": PRODUCTION_EVIDENCE_SCOPE,
    "dense_source": "dense_text",
    "sparse_source": "sparse_text",
    "item_facet_export_option": PRODUCTION_FACET_EXPORT_OPTION,
    "production_core_facet_mask": CORE_GRAPH_MASK_DESCRIPTION,
    "production_brand_facet_mask": BRAND_GRAPH_MASK_DESCRIPTION,
    "production_global_review_facet_mask": GLOBAL_REVIEW_GRAPH_MASK_DESCRIPTION,
    "historical_review_reputation_enabled": True,
    "review_reputation_graph_enabled": True,
    "brand_in_functional_graph": False,
    "brand_graph_enabled": True,
    "brand_in_retrieval_text": True,
    "brand_in_profile_source_text": True,
    "brand_in_synthetic_query": False,
    "raw_review_text_exported": False,
    "facet_type_to_role": COMMON_FACET_TYPE_TO_ROLE,
    "generic_category_anchors": sorted(COMMON_GENERIC_CATEGORY_ANCHORS),
    "generic_utility_tokens": sorted(COMMON_GENERIC_UTILITY_TOKENS),
    "context_dependent_utility_tokens": sorted(COMMON_CONTEXT_DEPENDENT_UTILITY_TOKENS),
    "specific_phrase_exceptions": sorted(COMMON_SPECIFIC_PHRASE_EXCEPTIONS),
    "possible_multiword_entity_fragment_tokens": sorted(HERBAL_MULTIWORD_ENTITY_FRAGMENT_TOKENS),
    "functional_facet_roles": sorted(COMMON_FUNCTIONAL_FACET_ROLES),
    "brand_policy": COMMON_BRAND_POLICY,
    "functional_facet_definition": COMMON_FUNCTIONAL_FACET_DEFINITION,
    "brand_diagnostic_metrics": COMMON_BRAND_DIAGNOSTIC_METRICS,
}


In [5]:
# ==== Define Normalization and Safety Checks ====
def normalize_space(value):
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    text = str(value).replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", text).strip()


def normalize_entity_value(value):
    text = normalize_space(value).lower()
    text = re.sub(r"[^a-z0-9가-힣\s_\-+/]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def split_values(value):
    text = normalize_space(value)
    if not text:
        return []

    values = []
    for part in re.split(r"\s*\|\s*|\s*;\s*|\s*,\s*", text):
        part = normalize_space(part)
        if part:
            values.append(part)

    output = []
    seen = set()
    for value in values:
        key = normalize_entity_value(value)
        if key and key not in seen:
            seen.add(key)
            output.append(value)
    return output


def make_entity_node_id(facet_type, value):
    return f"entity::{facet_type}::{normalize_entity_value(value)}"


def non_empty_ratio(series):
    if len(series) == 0:
        return 0.0
    return float(series.fillna("").astype(str).str.strip().ne("").mean())


def to_review_timestamp_ms(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        parsed = pd.to_datetime(series, utc=True, errors="coerce")
        return (parsed.astype("int64") // 1_000_000).where(parsed.notna(), np.nan)

    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.dropna().empty:
        parsed = pd.to_datetime(series, utc=True, errors="coerce")
        return (parsed.astype("int64") // 1_000_000).where(parsed.notna(), np.nan)

    if numeric.dropna().max() < 10**11:
        numeric = numeric * 1000
    return numeric


def read_parquet_columns(path, required_columns, optional_columns=None):
    import pyarrow.parquet as pq

    optional_columns = optional_columns or []
    available_columns = set(pq.ParquetFile(path).schema.names)
    missing_required = [column for column in required_columns if column not in available_columns]
    if missing_required:
        raise RuntimeError(f"Missing required parquet columns: {missing_required}")
    columns = required_columns + [
        column for column in optional_columns
        if column in available_columns
    ]
    return pd.read_parquet(path, columns=list(dict.fromkeys(columns)))


def assert_no_leakage_columns(df, frame_name):
    exact_forbidden = {
        "target_review_text",
        "heldout_review_text",
        "review_text",
        "review_body",
        "raw_review_text",
        "rating",
        "average_rating",
        "rating_number",
        "rating_count",
        "rating_num",
        "sentiment",
        "helpful_vote",
        "helpful_votes",
        "total_vote",
        "prompt",
        "response",
        "llm_response",
        "query_evidence",
    }
    forbidden_prefixes = (
        "target_review_",
        "heldout_review_",
        "raw_review_",
        "review_text_",
        "review_body_",
        "review_evidence_",
    )
    blocked = [
        str(column)
        for column in df.columns
        if str(column) in exact_forbidden
        or any(str(column).startswith(prefix) for prefix in forbidden_prefixes)
    ]
    if blocked:
        raise RuntimeError(f"{frame_name} contains forbidden leakage columns: {blocked}")


def assert_allowed_output_path(path):
    path_str = str(path).lower()
    blocked_terms = [
        "target_review_text",
        "heldout_review_text",
        "raw_review",
        "review_body",
        "full_history_user",
        "user_profile",
        "user_anchor",
    ]
    if any(term in path_str for term in blocked_terms):
        raise RuntimeError(f"Unsafe output path: {path}")


In [6]:
# ==== Load and Validate the Canonical Item Schema ====
schema_df = pd.read_parquet(SCHEMA_PATH).copy()

required_schema_columns = [
    "parent_asin",
    "title",
    "facet_brand_text",
    "brand_facet_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "generic_category_anchor_text",
    "generic_utility_token_text",
    "context_dependent_utility_token_text",
    "canonical_metadata_text",
    "canonical_item_metadata_source_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "review_reputation_only_text",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_safe_facet_text",
    "profile_source_text_dedup_seed",
    "brand_retrieval_enabled",
    "brand_profile_enabled",
    "brand_query_enabled",
    "evidence_scope",
    "historical_review_reputation_enabled",
    "facet_policy_version",
    "brand_policy",
    "identifier_policy",
    *CATEGORY_REQUIRED_FACET_COLS,
]
missing_schema_columns = [
    column for column in required_schema_columns
    if column not in schema_df.columns
]
if missing_schema_columns:
    raise RuntimeError(
        "Notebook 02 schema is missing required Global Review retrieval columns: "
        f"{missing_schema_columns}"
    )

if schema_df["parent_asin"].isna().any():
    raise RuntimeError("Schema contains null parent_asin values.")
schema_df["parent_asin"] = schema_df["parent_asin"].astype(str).str.strip()
if schema_df["parent_asin"].eq("").any():
    raise RuntimeError("Schema contains empty parent_asin values.")
if schema_df["parent_asin"].duplicated().any():
    raise RuntimeError("Schema parent_asin must be unique.")

policy_values = {
    "facet_policy_version": sorted(schema_df["facet_policy_version"].dropna().astype(str).unique().tolist()),
    "brand_policy": sorted(schema_df["brand_policy"].dropna().astype(str).unique().tolist()),
    "identifier_policy": sorted(schema_df["identifier_policy"].dropna().astype(str).unique().tolist()),
}
if policy_values["facet_policy_version"] != [EXPECTED_FACET_POLICY_VERSION]:
    raise RuntimeError(f"Unexpected facet_policy_version values: {policy_values['facet_policy_version']}")
if policy_values["brand_policy"] != [EXPECTED_BRAND_POLICY]:
    raise RuntimeError(f"Unexpected brand_policy values: {policy_values['brand_policy']}")
if policy_values["identifier_policy"] != [EXPECTED_IDENTIFIER_POLICY]:
    raise RuntimeError(f"Unexpected identifier_policy values: {policy_values['identifier_policy']}")
if not schema_df["evidence_scope"].eq(PRODUCTION_EVIDENCE_SCOPE).all():
    raise RuntimeError("Notebook 02 evidence_scope must match the Global Review production contract.")
if not schema_df["historical_review_reputation_enabled"].eq(True).all():
    raise RuntimeError("Historical review-reputation must be enabled for the Global Review production pipeline.")
if not schema_df["brand_retrieval_enabled"].eq(True).all():
    raise RuntimeError("brand_retrieval_enabled must be True.")
if not schema_df["brand_profile_enabled"].eq(True).all():
    raise RuntimeError("brand_profile_enabled must be True.")
if not schema_df["brand_query_enabled"].eq(False).all():
    raise RuntimeError("brand_query_enabled must be False.")

schema_df = schema_df.drop(
    columns=[column for column in ["average_rating", "rating_number"] if column in schema_df.columns]
)
assert_no_leakage_columns(schema_df, "schema_df")

discontinued_mask = pd.Series(False, index=schema_df.index)
for column in ["is_discontinued_norm", "is_discontinued"]:
    if column in schema_df.columns:
        discontinued_mask |= (
            schema_df[column]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
            .isin({"yes", "true", "1"})
        )

active_items_df = schema_df.loc[~discontinued_mask].copy().reset_index(drop=True)
discontinued_removed_count = int(discontinued_mask.sum())
if active_items_df.empty:
    raise RuntimeError("No active items remain after discontinued filtering.")

print("Rows:", len(active_items_df))
print("Validation: Global Review schema contract passed")


Rows: 27253
Validation: Global Review schema contract passed


In [7]:
# ==== Build Dense and Sparse Item Documents ====
core_doc_columns = [
    "parent_asin",
    "title",
    "facet_brand_text",
    "brand_facet_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "generic_category_anchor_text",
    "generic_utility_token_text",
    "context_dependent_utility_token_text",
    "canonical_metadata_text",
    "canonical_item_metadata_source_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "canonical_retrieval_text",
    "profile_safe_facet_text",
    "profile_source_text_dedup_seed",
    "brand_retrieval_enabled",
    "brand_profile_enabled",
    "brand_query_enabled",
    "evidence_scope",
    "historical_review_reputation_enabled",
    *CATEGORY_REQUIRED_FACET_COLS,
]
optional_signal_columns = [
    "historical_review_reputation_text",
    "review_reputation_facet_text",
    "review_reputation_only_text",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
]
optional_doc_columns = [
    column for column in [
        "is_discontinued_norm",
        "is_discontinued",
        *optional_signal_columns,
    ]
    if column in active_items_df.columns
]
item_docs = active_items_df[
    list(dict.fromkeys(core_doc_columns + optional_doc_columns))
].copy()

item_docs["brand_facet_text"] = (
    item_docs["brand_facet_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["specific_query_safe_facet_text"] = (
    item_docs["specific_query_safe_facet_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["functional_facet_text"] = (
    item_docs["functional_facet_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["profile_safe_facet_text"] = (
    item_docs["profile_safe_facet_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["generic_anchor_text"] = (
    item_docs["generic_category_anchor_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["generic_utility_text"] = (
    item_docs["generic_utility_token_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["context_dependent_utility_text"] = (
    item_docs["context_dependent_utility_token_text"].fillna("").astype(str).map(normalize_space)
)

review_signal_source = (
    "review_reputation_only_text"
    if "review_reputation_only_text" in item_docs.columns
    else "review_reputation_facet_text"
    if "review_reputation_facet_text" in item_docs.columns
    else None
)
item_docs["review_derived_signal_text"] = (
    item_docs[review_signal_source].fillna("").astype(str).map(normalize_space)
    if review_signal_source
    else ""
)

item_docs["dense_text_core"] = (
    item_docs["canonical_text_dense_core"].fillna("").astype(str).map(normalize_space)
)
item_docs["sparse_text_core"] = (
    item_docs["canonical_text_sparse_core"].fillna("").astype(str).map(normalize_space)
)
# Production retrieval text combines catalog evidence with frozen review-derived item signals.
item_docs["canonical_retrieval_text"] = (
    item_docs["canonical_retrieval_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["canonical_text_dense"] = (
    item_docs["canonical_text_dense"].fillna("").astype(str).map(normalize_space)
)
item_docs["canonical_text_sparse"] = (
    item_docs["canonical_text_sparse"].fillna("").astype(str).map(normalize_space)
)
item_docs["dense_text"] = item_docs["canonical_text_dense"]
item_docs["sparse_text"] = item_docs["canonical_text_sparse"]
item_docs["dense_text_v2"] = item_docs["dense_text"]
item_docs["bm25_text"] = item_docs["sparse_text"]
item_docs["bm25_text_v2"] = item_docs["sparse_text"]
item_docs["canonical_text"] = item_docs["canonical_retrieval_text"]

if len(item_docs) <= 0:
    raise RuntimeError("Retained item-doc count must be greater than zero.")
for column in ["dense_text", "sparse_text"]:
    if item_docs[column].eq("").any():
        raise RuntimeError(f"{column} must be non-empty for every retained item.")
if not item_docs["review_derived_signal_text"].fillna("").astype(str).str.strip().ne("").any():
    raise RuntimeError("Global Review item documents require non-empty historical review signals.")
brand_in_functional_text_rows = int(
    item_docs.apply(
        lambda row: bool(
            {normalize_entity_value(value) for value in split_values(row.get("brand_facet_text", ""))}
            & {normalize_entity_value(value) for value in split_values(row.get("functional_facet_text", ""))}
        ),
        axis=1,
    ).sum()
)
if brand_in_functional_text_rows:
    raise RuntimeError("functional_facet_text contains brand values.")

def contains_all_brand_values(container_text, brand_text):
    brand_values = {
        normalize_entity_value(value)
        for value in split_values(brand_text)
        if normalize_entity_value(value)
    }
    normalized_container = normalize_entity_value(container_text)
    return all(value in normalized_container for value in brand_values)


brand_rows_mask = item_docs["brand_facet_text"].fillna("").astype(str).str.strip().ne("")
brand_missing_dense_rows = int(
    item_docs.loc[brand_rows_mask].apply(
        lambda row: not contains_all_brand_values(
            row.get("dense_text", ""),
            row.get("brand_facet_text", ""),
        ),
        axis=1,
    ).sum()
)
brand_missing_sparse_rows = int(
    item_docs.loc[brand_rows_mask].apply(
        lambda row: not contains_all_brand_values(
            row.get("sparse_text", ""),
            row.get("brand_facet_text", ""),
        ),
        axis=1,
    ).sum()
)
brand_missing_profile_rows = int(
    item_docs.loc[brand_rows_mask].apply(
        lambda row: not contains_all_brand_values(
            row.get("profile_safe_facet_text", ""),
            row.get("brand_facet_text", ""),
        ),
        axis=1,
    ).sum()
)
if brand_missing_dense_rows or brand_missing_sparse_rows:
    raise RuntimeError(
        "Brand must be present in dense and sparse production item text. "
        f"Dense missing: {brand_missing_dense_rows}; sparse missing: {brand_missing_sparse_rows}."
    )
if brand_missing_profile_rows:
    raise RuntimeError(
        f"Brand must be present in profile_safe_facet_text. Missing rows: {brand_missing_profile_rows}."
    )

item_docs_dense = item_docs[[
    "parent_asin",
    "title",
    "dense_text",
    "dense_text_v2",
    "dense_text_core",
    "canonical_text_dense",
    "canonical_retrieval_text_core",
    "canonical_retrieval_text",
    "functional_facet_text",
    "specific_query_safe_facet_text",
    "brand_facet_text",
    "profile_safe_facet_text",
    "generic_anchor_text",
    "generic_utility_text",
    "context_dependent_utility_text",
    "review_derived_signal_text",
    "profile_source_text_dedup_seed",
]].copy()

item_docs_sparse = item_docs[[
    "parent_asin",
    "title",
    "sparse_text",
    "bm25_text",
    "bm25_text_v2",
    "sparse_text_core",
    "canonical_text_sparse",
    "canonical_retrieval_text_core",
    "canonical_retrieval_text",
    "functional_facet_text",
    "specific_query_safe_facet_text",
    "brand_facet_text",
    "profile_safe_facet_text",
    "generic_anchor_text",
    "generic_utility_text",
    "context_dependent_utility_text",
    "review_derived_signal_text",
    "profile_source_text_dedup_seed",
]].copy()

for frame_name, frame in [
    ("item_docs", item_docs),
    ("item_docs_dense", item_docs_dense),
    ("item_docs_sparse", item_docs_sparse),
]:
    duplicate_columns = frame.columns[frame.columns.duplicated()].tolist()
    if duplicate_columns:
        raise RuntimeError(f"{frame_name} contains duplicated columns: {duplicate_columns}")
    assert_no_leakage_columns(frame, frame_name)

print("Rows:", len(item_docs))
print("Validation: Global Review item documents passed")


Rows: 27253
Validation: Global Review item documents passed


In [8]:
# ==== Construct Long-Form Item Facets ====
METADATA_FACET_SPECS = [
    ("facet_brand_text", "brand", "HAS_BRAND"),
    ("facet_category_text", "category", "HAS_CATEGORY"),
    ("facet_form_text", "form", "HAS_FORM"),
    ("facet_ingredient_text", "ingredient", "HAS_INGREDIENT"),
    ("facet_benefit_text", "benefit", "HAS_BENEFIT"),
    ("facet_claim_text", "claim", "HAS_CLAIM"),
]

REVIEW_REPUTATION_FACET_SPECS = [
    ("review_reputation_concern_text", "review_reputation_concern", "HAS_REVIEW_REPUTATION_CONCERN"),
    ("review_reputation_skin_type_text", "review_reputation_skin_type", "HAS_REVIEW_REPUTATION_SKIN_TYPE"),
    ("review_reputation_benefit_text", "review_reputation_benefit", "HAS_REVIEW_REPUTATION_BENEFIT"),
    ("review_reputation_ingredient_text", "review_reputation_ingredient", "HAS_REVIEW_REPUTATION_INGREDIENT"),
    ("review_reputation_product_type_or_form_texture_text", "review_reputation_product_form_texture", "HAS_REVIEW_REPUTATION_PRODUCT_FORM_TEXTURE"),
    ("historical_review_ingredient_text", "review_reputation_ingredient", "HAS_REVIEW_REPUTATION_INGREDIENT"),
    ("historical_review_benefit_text", "review_reputation_benefit", "HAS_REVIEW_REPUTATION_BENEFIT"),
    ("historical_review_form_text", "review_reputation_form", "HAS_REVIEW_REPUTATION_FORM"),
    ("historical_review_claim_diet_text", "review_reputation_claim_diet", "HAS_REVIEW_REPUTATION_CLAIM_DIET"),
    ("historical_review_flavor_text", "review_reputation_flavor", "HAS_REVIEW_REPUTATION_FLAVOR"),
]

metadata_specs_used = list(dict.fromkeys([
    spec for spec in METADATA_FACET_SPECS + CATEGORY_OPTIONAL_FACET_SPECS
    if spec[0] in active_items_df.columns
]))
review_specs_used = [
    spec for spec in REVIEW_REPUTATION_FACET_SPECS
    if spec[0] in active_items_df.columns
]
if not review_specs_used and "review_reputation_facet_text" in active_items_df.columns:
    review_specs_used = [
        ("review_reputation_facet_text", "review_reputation", "HAS_REVIEW_REPUTATION")
    ]

FACET_SPECS = metadata_specs_used + review_specs_used
facet_rows = []
for row in active_items_df.itertuples(index=False):
    row_dict = row._asdict()
    parent_asin = normalize_space(row_dict.get("parent_asin"))
    title = normalize_space(row_dict.get("title"))

    for source_column, facet_type, edge_type in FACET_SPECS:
        for value in split_values(row_dict.get(source_column)):
            value_norm = normalize_entity_value(value)
            if value_norm:
                facet_rows.append({
                    "parent_asin": parent_asin,
                    "title": title,
                    "source_column": source_column,
                    "facet_type": facet_type,
                    "facet_value": value,
                    "facet_value_norm": value_norm,
                    "entity_node_id": make_entity_node_id(facet_type, value),
                    "edge_type": edge_type,
                })

item_facets = pd.DataFrame(facet_rows)
if item_facets.empty:
    raise RuntimeError("No item facets were generated.")

item_facets = (
    item_facets
    .drop_duplicates(["parent_asin", "facet_type", "facet_value_norm", "source_column"])
    .sort_values(["parent_asin", "facet_type", "facet_value_norm", "source_column"])
    .reset_index(drop=True)
)
assert_no_leakage_columns(item_facets, "item_facets")


In [9]:
# ==== Classify Facets and Apply Retrieval Masks ====
def common_role_from_facet_type(facet_type):
    facet_type = normalize_space(facet_type).lower()
    if facet_type in COMMON_FACET_TYPE_TO_ROLE:
        return COMMON_FACET_TYPE_TO_ROLE[facet_type]
    if facet_type.startswith("review_reputation") or facet_type.startswith("historical_review"):
        return "review_derived_signal"
    return "category_specific_other"

metadata_source_columns = {spec[0] for spec in metadata_specs_used}
item_facets["facet_role"] = item_facets["facet_type"].map(common_role_from_facet_type)
item_facets["facet_family"] = item_facets["facet_role"]

brand_values_by_item = {
    row.parent_asin: {
        normalize_entity_value(value)
        for value in split_values(row.brand_facet_text)
    }
    for row in item_docs[
        ["parent_asin", "brand_facet_text"]
    ].itertuples(index=False)
}

item_facets["is_row_brand_value"] = item_facets.apply(
    lambda row: (
        row["facet_value_norm"]
        in brand_values_by_item.get(row["parent_asin"], set())
    ),
    axis=1,
)

item_facets["is_brand"] = (
    item_facets["facet_role"].eq("brand")
    | item_facets["is_row_brand_value"]
)
item_facets["is_review_derived"] = item_facets["facet_role"].eq("review_derived_signal")
item_facets["is_metadata_facet_source"] = item_facets["source_column"].isin(metadata_source_columns)
item_facets["is_disallowed_nonfacet_source"] = item_facets["source_column"].fillna("").astype(str).str.lower().map(
    lambda value: any(term in value for term in FORBIDDEN_FACET_SOURCE_TERMS)
)
item_facets["is_generic_category_anchor"] = (
    item_facets["facet_value_norm"].fillna("").astype(str).str.lower().isin(COMMON_GENERIC_CATEGORY_ANCHORS)
)
item_facets["is_generic_utility_token"] = (
    item_facets["facet_value_norm"].fillna("").astype(str).str.lower().isin(COMMON_GENERIC_UTILITY_TOKENS)
)
item_facets["is_context_dependent_utility_token"] = (
    item_facets["facet_value_norm"].fillna("").astype(str).str.lower().isin(COMMON_CONTEXT_DEPENDENT_UTILITY_TOKENS)
)
item_facets["is_specific_phrase_exception"] = (
    item_facets["facet_value_norm"].fillna("").astype(str).str.lower().isin(COMMON_SPECIFIC_PHRASE_EXCEPTIONS)
)
item_facets["is_possible_entity_fragment"] = (
    item_facets["facet_role"].eq("ingredient_or_composition")
    & item_facets["facet_value_norm"].fillna("").astype(str).str.lower().isin(HERBAL_MULTIWORD_ENTITY_FRAGMENT_TOKENS)
)
item_facets["is_specific_facet_phrase"] = (
    item_facets["facet_role"].isin(COMMON_FUNCTIONAL_FACET_ROLES)
    & ~item_facets["is_brand"]
    & ~item_facets["is_review_derived"]
    & (
        item_facets["is_specific_phrase_exception"]
        | (
            ~item_facets["is_generic_category_anchor"]
            & ~item_facets["is_generic_utility_token"]
            & ~item_facets["is_context_dependent_utility_token"]
        )
    )
)
item_facets["is_product_functional_facet"] = item_facets["is_specific_facet_phrase"]
item_facets["is_query_safe"] = item_facets["is_product_functional_facet"]
item_facets["is_core_graph_facet"] = (
    item_facets["is_product_functional_facet"]
    & item_facets["is_query_safe"]
    & ~item_facets["is_brand"]
    & ~item_facets["is_review_derived"]
    & ~item_facets["is_generic_category_anchor"]
    & ~item_facets["is_generic_utility_token"]
    & ~item_facets["is_context_dependent_utility_token"]
    & item_facets["is_metadata_facet_source"]
    & ~item_facets["is_disallowed_nonfacet_source"]
)
item_facets["is_brand_graph_facet"] = (
    item_facets["facet_role"].eq("brand")
    & item_facets["is_brand"]
    & ~item_facets["is_review_derived"]
    & ~item_facets["is_generic_category_anchor"]
    & ~item_facets["is_generic_utility_token"]
    & ~item_facets["is_context_dependent_utility_token"]
    & item_facets["is_metadata_facet_source"]
    & ~item_facets["is_disallowed_nonfacet_source"]
)
item_facets["is_retrieval_safe"] = (
    item_facets["is_core_graph_facet"]
    | item_facets["is_brand_graph_facet"]
    | (
        item_facets["is_review_derived"]
        & ~item_facets["is_brand"]
        & ~item_facets["is_generic_category_anchor"]
        & ~item_facets["is_generic_utility_token"]
        & ~item_facets["is_context_dependent_utility_token"]
        & ~item_facets["is_disallowed_nonfacet_source"]
    )
)
item_facets["is_profile_safe"] = (
    item_facets["is_core_graph_facet"]
    | item_facets["is_brand_graph_facet"]
)
item_facets["is_global_review_graph_facet"] = item_facets["is_retrieval_safe"]
item_facets["common_framework_version"] = COMMON_RETRIEVAL_ARTIFACT_FRAMEWORK_VERSION

core_item_facets = item_facets.loc[item_facets["is_core_graph_facet"]].copy().reset_index(drop=True)
brand_item_facets = item_facets.loc[item_facets["is_brand_graph_facet"]].copy().reset_index(drop=True)
global_review_item_facets = item_facets.loc[
    item_facets["is_global_review_graph_facet"]
].copy().reset_index(drop=True)
review_item_facets = global_review_item_facets.loc[
    global_review_item_facets["is_review_derived"]
].copy().reset_index(drop=True)

if core_item_facets.empty:
    raise RuntimeError("Metadata product-functional facet rows must be greater than zero.")
if brand_item_facets.empty:
    raise RuntimeError("Brand facet rows must be greater than zero.")
if review_item_facets.empty:
    raise RuntimeError("Global Review requires non-empty historical review facet rows.")

if item_facets.loc[item_facets["is_brand"], "is_product_functional_facet"].any():
    raise RuntimeError("Brand rows must not be product-functional facets.")
if item_facets.loc[item_facets["is_brand"], "is_query_safe"].any():
    raise RuntimeError("Brand rows must not be query-safe.")
if not item_facets.loc[item_facets["is_brand_graph_facet"], "is_retrieval_safe"].all():
    raise RuntimeError("Every brand graph facet must be retrieval-safe.")
if not item_facets.loc[item_facets["is_brand_graph_facet"], "is_profile_safe"].all():
    raise RuntimeError("Every brand graph facet must be profile-safe.")
if item_facets.loc[item_facets["is_brand_graph_facet"], "is_product_functional_facet"].any():
    raise RuntimeError("Brand graph facets must remain separate from product-functional facets.")
if item_facets.loc[item_facets["is_review_derived"], "is_product_functional_facet"].any():
    raise RuntimeError("Review-derived rows must not be product-functional facets.")
if item_facets.loc[item_facets["is_review_derived"], "is_query_safe"].any():
    raise RuntimeError("Review-derived rows must not be query-safe.")
if item_facets.loc[item_facets["is_generic_category_anchor"], "is_specific_facet_phrase"].any():
    raise RuntimeError("Generic category anchors must not be specific facet phrases.")
if item_facets.loc[item_facets["is_generic_utility_token"], "is_specific_facet_phrase"].any():
    raise RuntimeError("Standalone generic utility tokens must not be specific facet phrases.")
if item_facets.loc[
    item_facets["is_context_dependent_utility_token"] & ~item_facets["is_specific_phrase_exception"],
    "is_specific_facet_phrase",
].any():
    raise RuntimeError("Standalone context-dependent utility tokens must not be specific facet phrases.")

facet_counts = (
    item_facets.groupby("facet_type").size().sort_values(ascending=False).astype(int).to_dict()
)
facet_vocab = {
    facet_type: sorted(values)
    for facet_type, values in (
        global_review_item_facets
        .groupby("facet_type")["facet_value_norm"]
        .apply(lambda series: set(series.dropna().astype(str)))
        .items()
    )
}
facet_vocab_df = (
    global_review_item_facets[["facet_type", "facet_value_norm"]]
    .drop_duplicates()
    .sort_values(["facet_type", "facet_value_norm"])
    .reset_index(drop=True)
)

facet_role_coverage_df = (
    item_facets
    .groupby(["facet_role", "facet_type"], dropna=False)
    .agg(
        facet_rows=("facet_value_norm", "size"),
        item_count=("parent_asin", "nunique"),
        unique_value_count=("facet_value_norm", "nunique"),
        query_safe_rows=("is_query_safe", "sum"),
        brand_rows=("is_brand", "sum"),
        product_functional_rows=("is_product_functional_facet", "sum"),
        review_derived_rows=("is_review_derived", "sum"),
        generic_anchor_rows=("is_generic_category_anchor", "sum"),
        generic_utility_rows=("is_generic_utility_token", "sum"),
        context_dependent_utility_rows=("is_context_dependent_utility_token", "sum"),
        possible_entity_fragment_rows=("is_possible_entity_fragment", "sum"),
        specific_facet_phrase_rows=("is_specific_facet_phrase", "sum"),
        core_graph_rows=("is_core_graph_facet", "sum"),
        brand_graph_rows=("is_brand_graph_facet", "sum"),
        retrieval_safe_rows=("is_retrieval_safe", "sum"),
        profile_safe_rows=("is_profile_safe", "sum"),
        global_review_graph_rows=("is_global_review_graph_facet", "sum"),
    )
    .reset_index()
    .sort_values(["facet_role", "facet_rows"], ascending=[True, False])
)
facet_role_coverage_df["coverage_rate_with_brand"] = (
    facet_role_coverage_df["item_count"] / max(1, int(active_items_df["parent_asin"].nunique()))
)
facet_role_coverage_df["coverage_rate_no_brand"] = np.where(
    facet_role_coverage_df["facet_role"].eq("brand"),
    np.nan,
    facet_role_coverage_df["coverage_rate_with_brand"],
)

print("Rows:", len(item_facets))
print("Rows:", len(core_item_facets))
print("Rows:", len(review_item_facets))
print("Rows:", len(global_review_item_facets))
print("Validation: Global Review facet mask passed")


Rows: 200477
Rows: 102099
Rows: 66885
Rows: 196225
Validation: Global Review facet mask passed


In [10]:
# ==== Build the Item–Facet Graph ====
graph_long = global_review_item_facets[[
    "parent_asin",
    "title",
    "entity_node_id",
    "edge_type",
    "facet_type",
    "facet_value",
    "facet_value_norm",
    "source_column",
    "facet_role",
    "facet_family",
    "is_brand",
    "is_review_derived",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_product_functional_facet",
    "is_query_safe",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]].copy()

graph_long["product_node_id"] = "item::" + graph_long["parent_asin"].astype(str)
graph_long["source_node_id"] = graph_long["product_node_id"]
graph_long["target_node_id"] = graph_long["entity_node_id"]
graph_long["source_node_type"] = "item"
graph_long["target_node_type"] = "entity"

graph_long = graph_long[[
    "parent_asin",
    "title",
    "product_node_id",
    "entity_node_id",
    "source_node_id",
    "target_node_id",
    "source_node_type",
    "target_node_type",
    "edge_type",
    "facet_type",
    "facet_value",
    "facet_value_norm",
    "source_column",
    "facet_role",
    "facet_family",
    "is_brand",
    "is_review_derived",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_product_functional_facet",
    "is_query_safe",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]].drop_duplicates().reset_index(drop=True)

graph_edges = graph_long[[
    "source_node_id",
    "target_node_id",
    "source_node_type",
    "target_node_type",
    "edge_type",
    "facet_type",
    "facet_value_norm",
    "source_column",
    "facet_role",
    "facet_family",
    "is_brand",
    "is_review_derived",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_product_functional_facet",
    "is_query_safe",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]].copy()
graph_edges["edge_id"] = (
    graph_edges["source_node_id"].astype(str)
    + "||"
    + graph_edges["edge_type"].astype(str)
    + "||"
    + graph_edges["target_node_id"].astype(str)
)
graph_edges = graph_edges.drop_duplicates("edge_id").reset_index(drop=True)

item_kg_edges = graph_edges.rename(
    columns={"source_node_id": "product_node_id", "target_node_id": "entity_node_id"}
)[[
    "edge_id",
    "product_node_id",
    "entity_node_id",
    "edge_type",
    "facet_type",
    "facet_value_norm",
    "source_column",
    "facet_role",
    "facet_family",
    "is_brand",
    "is_review_derived",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_product_functional_facet",
    "is_query_safe",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]].copy()

product_node_cols = [
    "parent_asin",
    "title",
    "brand_facet_text",
    "profile_safe_facet_text",
    "query_safe_facet_text",
    "functional_facet_text",
]
product_nodes = item_docs[product_node_cols].copy()
product_nodes["node_id"] = "item::" + product_nodes["parent_asin"].astype(str)
product_nodes["node_type"] = "item"
product_nodes = product_nodes[["node_id", "node_type"] + product_node_cols].drop_duplicates("node_id").reset_index(drop=True)

entity_nodes = graph_long[["entity_node_id", "facet_type", "facet_value_norm"]].drop_duplicates().copy()
entity_nodes = entity_nodes.rename(columns={"entity_node_id": "node_id"})
entity_nodes["node_type"] = "entity"
entity_nodes = entity_nodes[["node_id", "node_type", "facet_type", "facet_value_norm"]].sort_values("node_id").reset_index(drop=True)

for frame_name, frame in [
    ("graph_long", graph_long),
    ("graph_edges", graph_edges),
    ("item_kg_edges", item_kg_edges),
    ("product_nodes", product_nodes),
    ("entity_nodes", entity_nodes),
]:
    assert_no_leakage_columns(frame, frame_name)

metadata_graph_edges = int(graph_edges["is_core_graph_facet"].fillna(False).astype(bool).sum())
brand_graph_edges = int(graph_edges["is_brand_graph_facet"].fillna(False).astype(bool).sum())
review_reputation_graph_edges = int(graph_edges["is_review_derived"].fillna(False).astype(bool).sum())
total_graph_edges = int(len(graph_edges))
generic_anchor_graph_edges = int(graph_edges["is_generic_category_anchor"].fillna(False).astype(bool).sum())
generic_utility_graph_edges = int(graph_edges["is_generic_utility_token"].fillna(False).astype(bool).sum())
context_utility_graph_edges = int(graph_edges["is_context_dependent_utility_token"].fillna(False).astype(bool).sum())
disallowed_source_graph_edges = int(graph_edges["is_disallowed_nonfacet_source"].fillna(False).astype(bool).sum())

if metadata_graph_edges <= 0:
    raise RuntimeError("Metadata graph edges must be greater than zero.")
if brand_graph_edges <= 0:
    raise RuntimeError("Brand graph edges must be greater than zero.")
if review_reputation_graph_edges <= 0:
    raise RuntimeError("Global Review requires historical review-reputation graph edges.")
if total_graph_edges != metadata_graph_edges + brand_graph_edges + review_reputation_graph_edges:
    raise RuntimeError(
        "Global Review graph must contain only metadata functional, brand, and review-derived edges."
    )
if generic_anchor_graph_edges or generic_utility_graph_edges:
    raise RuntimeError("Generic anchors or utility tokens must not enter the Global Review graph.")
if context_utility_graph_edges or disallowed_source_graph_edges:
    raise RuntimeError("Context utility or disallowed source rows must not enter the Global Review graph.")

metadata_edge_mask = graph_edges["is_core_graph_facet"].fillna(False).astype(bool)
brand_edge_mask = graph_edges["is_brand_graph_facet"].fillna(False).astype(bool)
review_edge_mask = graph_edges["is_review_derived"].fillna(False).astype(bool)
if not graph_edges.loc[metadata_edge_mask, "is_product_functional_facet"].all():
    raise RuntimeError("Every metadata graph edge must be product-functional.")
if not graph_edges.loc[metadata_edge_mask, "is_query_safe"].all():
    raise RuntimeError("Every metadata graph edge must be query-safe.")
if not graph_edges.loc[metadata_edge_mask, "is_metadata_facet_source"].all():
    raise RuntimeError("Every metadata graph edge must come from metadata facets.")
if graph_edges.loc[brand_edge_mask, "is_product_functional_facet"].any():
    raise RuntimeError("Brand graph edges must remain separate from product-functional facets.")
if graph_edges.loc[brand_edge_mask, "is_query_safe"].any():
    raise RuntimeError("Brand graph edges must not be query-safe.")
if not graph_edges.loc[brand_edge_mask, "is_profile_safe"].all():
    raise RuntimeError("Brand graph edges must be profile-safe.")
if not graph_edges.loc[brand_edge_mask, "is_retrieval_safe"].all():
    raise RuntimeError("Brand graph edges must be retrieval-safe.")
if graph_edges.loc[review_edge_mask, "is_product_functional_facet"].any():
    raise RuntimeError("Review-derived graph edges must remain separate from product-functional facets.")
if graph_edges.loc[review_edge_mask, "is_query_safe"].any():
    raise RuntimeError("Review-derived graph edges must not be marked query-safe.")

core_facet_filter_audit_df = pd.DataFrame([{
    "category_id": CATEGORY_ID,
    "item_facet_export_option": PRODUCTION_FACET_EXPORT_OPTION,
    "all_facet_rows": int(len(item_facets)),
    "core_facet_rows": int(len(core_item_facets)),
    "brand_facet_rows": int(len(brand_item_facets)),
    "review_facet_rows": int(len(review_item_facets)),
    "global_review_facet_rows": int(len(global_review_item_facets)),
    "core_facet_items": int(core_item_facets["parent_asin"].nunique()),
    "metadata_functional_facet_rows": int(len(core_item_facets)),
    "metadata_graph_edges": metadata_graph_edges,
    "brand_graph_edges": brand_graph_edges,
    "review_reputation_graph_edges": review_reputation_graph_edges,
    "global_review_graph_edges": total_graph_edges,
    "review_derived_rows_in_core_mask": int(core_item_facets["is_review_derived"].sum()),
    "brand_rows_in_core_mask": int(core_item_facets["is_brand"].sum()),
    "generic_anchor_rows_in_core_mask": int(core_item_facets["is_generic_category_anchor"].sum()),
    "generic_anchor_graph_edges": generic_anchor_graph_edges,
    "generic_utility_rows_in_core_mask": int(core_item_facets["is_generic_utility_token"].sum()),
    "generic_utility_graph_edges": generic_utility_graph_edges,
    "context_utility_graph_edges": context_utility_graph_edges,
    "disallowed_source_rows_in_core_mask": int(core_item_facets["is_disallowed_nonfacet_source"].sum()),
    "disallowed_source_graph_edges": disallowed_source_graph_edges,
    "status": "PASS",
}])

print("Rows:", len(graph_edges))
print("Validation: Global Review graph passed")


Rows: 196225
Validation: Global Review graph passed


In [11]:
# ==== Build Point-in-Time Review-Activity Indices ====
if not REVIEWS_PATH.exists():
    raise FileNotFoundError(f"Missing reviews file: {REVIEWS_PATH}")

review_required_cols = ["parent_asin", "timestamp"]
review_optional_cols = ["verified_purchase"]
reviews_temporal_raw = read_parquet_columns(REVIEWS_PATH, review_required_cols, review_optional_cols)

if "timestamp" not in reviews_temporal_raw.columns:
    raise RuntimeError("Processed reviews must contain timestamp for temporal artifacts.")

reviews_temporal = pd.DataFrame({
    "parent_asin": reviews_temporal_raw["parent_asin"].astype(str).map(normalize_space),
    "review_timestamp_ms": to_review_timestamp_ms(reviews_temporal_raw["timestamp"]),
})

if "verified_purchase" in reviews_temporal_raw.columns:
    reviews_temporal["verified_purchase"] = reviews_temporal_raw["verified_purchase"]

reviews_temporal = reviews_temporal[reviews_temporal["parent_asin"].ne("")].copy()
reviews_temporal["review_timestamp_ms"] = pd.to_numeric(reviews_temporal["review_timestamp_ms"], errors="coerce")
reviews_temporal = reviews_temporal.dropna(subset=["review_timestamp_ms"]).copy()
reviews_temporal["review_timestamp_ms"] = reviews_temporal["review_timestamp_ms"].astype("int64")
reviews_temporal["review_datetime"] = pd.to_datetime(reviews_temporal["review_timestamp_ms"], unit="ms", utc=True, errors="coerce")
reviews_temporal = reviews_temporal[reviews_temporal["review_datetime"].notna()].copy()

if reviews_temporal.empty:
    raise RuntimeError("No valid review timestamps are available for temporal artifacts.")

reviews_temporal = reviews_temporal.merge(
    active_items_df[["parent_asin"]].drop_duplicates(),
    on="parent_asin",
    how="inner",
)

reviews_temporal = reviews_temporal.sort_values(
    ["parent_asin", "review_timestamp_ms"],
    ascending=[True, True],
    kind="mergesort",
).reset_index(drop=True)
reviews_temporal["review_event_id"] = reviews_temporal.index.astype("int64")
reviews_temporal["review_date"] = reviews_temporal["review_datetime"].dt.floor("D")
reviews_temporal["review_count"] = 1

item_review_time_index = reviews_temporal[[
    "review_event_id",
    "parent_asin",
    "review_timestamp_ms",
    "review_datetime",
    "review_date",
    "review_count",
]].copy()

item_review_daily_counts = (
    reviews_temporal
    .groupby(["parent_asin", "review_date"], as_index=False)
    .agg(
        review_count=("review_count", "sum"),
        first_review_timestamp_ms=("review_timestamp_ms", "min"),
        last_review_timestamp_ms=("review_timestamp_ms", "max"),
    )
    .sort_values(["parent_asin", "review_date"], ascending=[True, True], kind="mergesort")
    .reset_index(drop=True)
)
item_review_daily_counts["cumulative_review_count"] = (
    item_review_daily_counts.groupby("parent_asin", sort=False)["review_count"].cumsum().astype("int64")
)

item_temporal_summary = (
    reviews_temporal
    .groupby("parent_asin", as_index=False)
    .agg(
        item_total_review_count=("review_count", "sum"),
        item_first_review_timestamp_ms=("review_timestamp_ms", "min"),
        item_last_review_timestamp_ms=("review_timestamp_ms", "max"),
        item_first_review_datetime=("review_datetime", "min"),
        item_last_review_datetime=("review_datetime", "max"),
    )
    .sort_values("parent_asin", kind="mergesort")
    .reset_index(drop=True)
)

metadata_item_facet_key = (
    core_item_facets.loc[core_item_facets["facet_type"].isin(TEMPORAL_FACET_TYPES), [
        "parent_asin",
        "facet_type",
        "facet_value_norm",
    ]]
    .dropna(subset=["parent_asin", "facet_type", "facet_value_norm"])
    .drop_duplicates()
    .copy()
)
metadata_item_facet_key = metadata_item_facet_key[
    metadata_item_facet_key["facet_value_norm"].astype(str).str.strip().ne("")
].copy()

entity_review_events = reviews_temporal[[
    "parent_asin",
    "review_timestamp_ms",
    "review_datetime",
    "review_date",
    "review_count",
]].merge(
    metadata_item_facet_key,
    on="parent_asin",
    how="inner",
)

entity_review_daily_counts = (
    entity_review_events
    .groupby(["facet_type", "facet_value_norm", "review_date"], as_index=False)
    .agg(
        review_count=("review_count", "sum"),
        item_count=("parent_asin", "nunique"),
        first_review_timestamp_ms=("review_timestamp_ms", "min"),
        last_review_timestamp_ms=("review_timestamp_ms", "max"),
    )
    .sort_values(["facet_type", "facet_value_norm", "review_date"], ascending=[True, True, True], kind="mergesort")
    .reset_index(drop=True)
)
entity_review_daily_counts["cumulative_review_count"] = (
    entity_review_daily_counts.groupby(["facet_type", "facet_value_norm"], sort=False)["review_count"].cumsum().astype("int64")
)

for frame_name, frame in [
    ("item_review_time_index", item_review_time_index),
    ("item_review_daily_counts", item_review_daily_counts),
    ("entity_review_daily_counts", entity_review_daily_counts),
    ("item_temporal_summary", item_temporal_summary),
]:
    assert_no_leakage_columns(frame, frame_name)

print("Rows:", len(item_review_time_index))
print("Validation: temporal metadata artifacts passed")


Rows: 590182
Validation: temporal metadata artifacts passed


In [12]:
# ==== Export Retrieval Artifacts ====
def write_parquet_with_aliases(df, canonical_path, alias_paths=None):
    canonical_path = Path(canonical_path)
    alias_paths = [Path(path) for path in (alias_paths or [])]
    for path in [canonical_path] + alias_paths:
        assert_allowed_output_path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(path, index=False)


for path in [
    ITEM_DOCS_DENSE_PATH,
    ITEM_DOCS_SPARSE_PATH,
    ITEM_DOCS_PATH,
    *ITEM_DOCS_COMPATIBILITY_PATHS,
    ITEMS_FACETS_PATH,
    *ITEMS_FACETS_COMPATIBILITY_PATHS,
    ITEM_GRAPH_LONG_PATH,
    ITEM_KG_EDGES_PATH,
    GRAPH_EDGES_PATH,
    PRODUCT_NODES_PATH,
    ENTITY_NODES_PATH,
    ACTIVE_SCHEMA_PATH,
    *ACTIVE_SCHEMA_COMPATIBILITY_PATHS,
    FACET_VOCAB_PATH,
    FACET_VOCAB_TABLE_PATH,
    FACET_COVERAGE_PATH,
    DOC_COVERAGE_PATH,
    CORE_FACET_FILTER_AUDIT_PATH,
    MANIFEST_PATH,
    ITEM_REVIEW_TIME_INDEX_PATH,
    ITEM_REVIEW_DAILY_COUNTS_PATH,
    ENTITY_REVIEW_DAILY_COUNTS_PATH,
    ITEM_TEMPORAL_SUMMARY_PATH,
    TEMPORAL_ARTIFACT_MANIFEST_PATH,
]:
    assert_allowed_output_path(path)

active_schema_export_cols = [
    column for column in active_items_df.columns
    if column != "identifier_diagnostic_text"
]
write_parquet_with_aliases(
    active_items_df[active_schema_export_cols],
    ACTIVE_SCHEMA_PATH,
    ACTIVE_SCHEMA_COMPATIBILITY_PATHS,
)
write_parquet_with_aliases(item_docs_dense, ITEM_DOCS_DENSE_PATH)
write_parquet_with_aliases(item_docs_sparse, ITEM_DOCS_SPARSE_PATH)
write_parquet_with_aliases(item_docs, ITEM_DOCS_PATH, ITEM_DOCS_COMPATIBILITY_PATHS)
write_parquet_with_aliases(item_facets, ITEMS_FACETS_PATH, ITEMS_FACETS_COMPATIBILITY_PATHS)
write_parquet_with_aliases(graph_long, ITEM_GRAPH_LONG_PATH)
write_parquet_with_aliases(item_kg_edges, ITEM_KG_EDGES_PATH)
write_parquet_with_aliases(graph_edges, GRAPH_EDGES_PATH)
write_parquet_with_aliases(product_nodes, PRODUCT_NODES_PATH)
write_parquet_with_aliases(entity_nodes, ENTITY_NODES_PATH)
write_parquet_with_aliases(item_review_time_index, ITEM_REVIEW_TIME_INDEX_PATH)
write_parquet_with_aliases(item_review_daily_counts, ITEM_REVIEW_DAILY_COUNTS_PATH)
write_parquet_with_aliases(entity_review_daily_counts, ENTITY_REVIEW_DAILY_COUNTS_PATH)
write_parquet_with_aliases(item_temporal_summary, ITEM_TEMPORAL_SUMMARY_PATH)

with open(FACET_VOCAB_PATH, "w", encoding="utf-8") as file:
    json.dump(facet_vocab, file, ensure_ascii=False, indent=2)
facet_vocab_df.to_parquet(FACET_VOCAB_TABLE_PATH, index=False)

facet_coverage_df = (
    item_facets.groupby(["facet_role", "facet_type"], dropna=False)
    .agg(
        row_count=("facet_value_norm", "size"),
        item_count=("parent_asin", "nunique"),
        unique_value_count=("facet_value_norm", "nunique"),
        core_graph_rows=("is_core_graph_facet", "sum"),
    )
    .reset_index()
    .sort_values(["row_count", "facet_type"], ascending=[False, True])
)
facet_coverage_df.to_csv(FACET_COVERAGE_PATH, index=False, encoding="utf-8-sig")

doc_coverage_df = pd.DataFrame([
    {
        "column": column,
        "non_empty_rows": int(item_docs[column].fillna("").astype(str).str.strip().ne("").sum()),
        "coverage_ratio": float(item_docs[column].fillna("").astype(str).str.strip().ne("").mean()),
    }
    for column in item_docs.columns
]).sort_values(["coverage_ratio", "column"], ascending=[False, True])
doc_coverage_df.to_csv(DOC_COVERAGE_PATH, index=False, encoding="utf-8-sig")
core_facet_filter_audit_df.to_csv(CORE_FACET_FILTER_AUDIT_PATH, index=False, encoding="utf-8-sig")

artifact_outputs = {
    "item_docs_dense": ITEM_DOCS_DENSE_PATH,
    "item_docs_sparse": ITEM_DOCS_SPARSE_PATH,
    "item_docs": ITEM_DOCS_PATH,
    "items_facets": ITEMS_FACETS_PATH,
    "item_graph_long": ITEM_GRAPH_LONG_PATH,
    "item_kg_edges": ITEM_KG_EDGES_PATH,
    "graph_edges": GRAPH_EDGES_PATH,
    "product_nodes": PRODUCT_NODES_PATH,
    "entity_nodes": ENTITY_NODES_PATH,
    "item_review_time_index": ITEM_REVIEW_TIME_INDEX_PATH,
    "item_review_daily_counts": ITEM_REVIEW_DAILY_COUNTS_PATH,
    "entity_review_daily_counts": ENTITY_REVIEW_DAILY_COUNTS_PATH,
    "item_temporal_summary": ITEM_TEMPORAL_SUMMARY_PATH,
    "temporal_artifact_manifest": TEMPORAL_ARTIFACT_MANIFEST_PATH,
    "facet_coverage": FACET_COVERAGE_PATH,
    "doc_coverage": DOC_COVERAGE_PATH,
    "core_facet_filter_audit": CORE_FACET_FILTER_AUDIT_PATH,
    "active_schema": ACTIVE_SCHEMA_PATH,
    "facet_vocab_json": FACET_VOCAB_PATH,
    "facet_vocab_table": FACET_VOCAB_TABLE_PATH,
}
for index, path in enumerate(ACTIVE_SCHEMA_COMPATIBILITY_PATHS, start=1):
    artifact_outputs[f"active_schema_alias_{index}"] = path
for index, path in enumerate(ITEM_DOCS_COMPATIBILITY_PATHS, start=1):
    artifact_outputs[f"item_docs_alias_{index}"] = path
for index, path in enumerate(ITEMS_FACETS_COMPATIBILITY_PATHS, start=1):
    artifact_outputs[f"items_facets_alias_{index}"] = path

temporal_manifest = {
    "stage": STAGE,
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "review_source_path": str(REVIEWS_PATH),
    "temporal_review_strict_rule": TEMPORAL_REVIEW_STRICT_RULE,
    "raw_review_text_loaded": False,
    "raw_review_text_exported": False,
    "target_review_text_used": False,
    "rating_used": False,
    "sentiment_used": False,
    "temporal_windows_days": TEMPORAL_WINDOWS_DAYS,
    "temporal_facet_types": sorted(entity_review_daily_counts["facet_type"].dropna().astype(str).unique().tolist()),
    "entity_facet_source": "core_item_facets",
    "output_paths": {
        "item_review_time_index": str(ITEM_REVIEW_TIME_INDEX_PATH),
        "item_review_daily_counts": str(ITEM_REVIEW_DAILY_COUNTS_PATH),
        "entity_review_daily_counts": str(ENTITY_REVIEW_DAILY_COUNTS_PATH),
        "item_temporal_summary": str(ITEM_TEMPORAL_SUMMARY_PATH),
        "temporal_artifact_manifest": str(TEMPORAL_ARTIFACT_MANIFEST_PATH),
    },
    "row_counts": {
        "item_review_time_index": int(len(item_review_time_index)),
        "item_review_daily_counts": int(len(item_review_daily_counts)),
        "entity_review_daily_counts": int(len(entity_review_daily_counts)),
        "item_temporal_summary": int(len(item_temporal_summary)),
    },
}
with open(TEMPORAL_ARTIFACT_MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(temporal_manifest, file, ensure_ascii=False, indent=2)

missing_outputs = [str(path) for path in artifact_outputs.values() if not Path(path).exists()]
if missing_outputs:
    raise RuntimeError(f"Expected output files were not written: {missing_outputs}")

print("Output:", ITEM_DOCS_PATH)
print("Output:", ITEMS_FACETS_PATH)
print("Output:", GRAPH_EDGES_PATH)
print("Output:", CORE_FACET_FILTER_AUDIT_PATH)
print("Rows:", len(item_docs))


Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/item_docs_herbal.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/items_facets_herbal.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/graph_edges_herbal.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_core_facet_filter_audit.csv
Rows: 27253


In [13]:
# ==== Validate Outputs and Write the Manifest ====
for frame_name, frame in [
    ("item_docs_dense", item_docs_dense),
    ("item_docs_sparse", item_docs_sparse),
    ("item_docs", item_docs),
]:
    if len(frame) <= 0:
        raise RuntimeError(f"{frame_name} must contain retained items.")
    if not frame["parent_asin"].astype(str).is_unique:
        raise RuntimeError(f"{frame_name} parent_asin must be unique.")

required_doc_columns = [
    "dense_text_core",
    "sparse_text_core",
    "dense_text",
    "sparse_text",
    "dense_text_v2",
    "bm25_text",
    "bm25_text_v2",
    "generic_anchor_text",
    "generic_utility_text",
    "context_dependent_utility_text",
    "specific_query_safe_facet_text",
    "brand_facet_text",
    "profile_safe_facet_text",
    "functional_facet_text",
]
missing_doc_columns = [column for column in required_doc_columns if column not in item_docs.columns]
if missing_doc_columns:
    raise RuntimeError(f"Missing required item-doc columns: {missing_doc_columns}")

for column in ["dense_text", "sparse_text"]:
    if item_docs[column].fillna("").astype(str).str.strip().eq("").any():
        raise RuntimeError(f"{column} must be non-empty for every retained item.")
if not item_docs["dense_text"].equals(item_docs["canonical_text_dense"]):
    raise RuntimeError("dense_text must equal canonical_text_dense.")
if not item_docs["sparse_text"].equals(item_docs["canonical_text_sparse"]):
    raise RuntimeError("sparse_text must equal canonical_text_sparse.")

review_derived_item_text_rows = int(
    item_docs["review_derived_signal_text"].fillna("").astype(str).str.strip().ne("").sum()
)
if review_derived_item_text_rows <= 0:
    raise RuntimeError("Review-derived item text must be non-empty for at least one active item.")

required_facet_flags = [
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_row_brand_value",
    "is_brand",
    "is_review_derived",
    "is_product_functional_facet",
    "is_query_safe",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]
missing_facet_flags = [column for column in required_facet_flags if column not in item_facets.columns]
if missing_facet_flags:
    raise RuntimeError(f"Missing common framework facet flags: {missing_facet_flags}")

if len(core_item_facets) <= 0:
    raise RuntimeError("Metadata functional facet rows must be greater than zero.")
if len(brand_item_facets) <= 0:
    raise RuntimeError("Brand facet rows must be greater than zero.")
if len(review_item_facets) <= 0:
    raise RuntimeError("Historical review facet rows must be greater than zero.")
historical_review_covered_items = int(review_item_facets["parent_asin"].nunique())
if historical_review_covered_items <= 0:
    raise RuntimeError("Historical review covered items must be greater than zero.")
if len(graph_edges) <= 0:
    raise RuntimeError("Global Review graph edges must be greater than zero.")
metadata_graph_edges = int(graph_edges["is_core_graph_facet"].fillna(False).astype(bool).sum())
historical_review_graph_edges = int(graph_edges["is_review_derived"].fillna(False).astype(bool).sum())
brand_graph_edges_used = int(graph_edges["is_brand_graph_facet"].fillna(False).astype(bool).sum())
raw_review_text_exported = any(
    ("raw_review" in str(column).lower()) or (str(column).lower() in {"review_text", "review_body"})
    for frame in [item_docs_dense, item_docs_sparse, item_docs, item_facets, graph_long, item_kg_edges, graph_edges]
    for column in frame.columns
)
if metadata_graph_edges <= 0:
    raise RuntimeError("Metadata graph edges must be greater than zero.")
if historical_review_graph_edges <= 0:
    raise RuntimeError("Historical review graph edges must be greater than zero.")
if brand_graph_edges_used <= 0:
    raise RuntimeError("Brand graph edges used by production must be greater than zero.")
if raw_review_text_exported:
    raise RuntimeError("Raw review text must not be exported.")
for column in [
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_disallowed_nonfacet_source",
]:
    if graph_edges[column].fillna(False).astype(bool).any():
        raise RuntimeError(f"Production graph contains disallowed rows: {column}")
if not graph_edges["is_global_review_graph_facet"].all():
    raise RuntimeError("Every production graph edge must satisfy the Global Review graph mask.")
if not graph_edges["is_brand_graph_facet"].fillna(False).astype(bool).any():
    raise RuntimeError("Production Global Review graph must include brand edges.")
if graph_edges.loc[graph_edges["is_brand_graph_facet"], "is_query_safe"].any():
    raise RuntimeError("Brand graph edges must not be query-safe.")
if not graph_edges.loc[graph_edges["is_brand_graph_facet"], "is_profile_safe"].all():
    raise RuntimeError("Brand graph edges must be profile-safe.")
if not graph_edges["is_review_derived"].fillna(False).astype(bool).any():
    raise RuntimeError("Production Global Review graph must include historical review-derived edges.")
if not graph_edges["source_node_type"].eq("item").all() or not graph_edges["target_node_type"].eq("entity").all():
    raise RuntimeError("graph_edges must be item-to-entity only.")

for frame_name, frame in [
    ("item_docs_dense", item_docs_dense),
    ("item_docs_sparse", item_docs_sparse),
    ("item_docs", item_docs),
    ("item_facets", item_facets),
    ("graph_long", graph_long),
    ("item_kg_edges", item_kg_edges),
    ("graph_edges", graph_edges),
    ("product_nodes", product_nodes),
    ("entity_nodes", entity_nodes),
    ("item_review_time_index", item_review_time_index),
    ("item_review_daily_counts", item_review_daily_counts),
    ("entity_review_daily_counts", entity_review_daily_counts),
    ("item_temporal_summary", item_temporal_summary),
]:
    assert_no_leakage_columns(frame, frame_name)

for frame_name, frame in {
    "item_review_time_index": item_review_time_index,
    "item_review_daily_counts": item_review_daily_counts,
    "entity_review_daily_counts": entity_review_daily_counts,
    "item_temporal_summary": item_temporal_summary,
}.items():
    if frame.empty:
        raise RuntimeError(f"{frame_name} must not be empty.")
if item_review_time_index["review_timestamp_ms"].isna().any():
    raise RuntimeError("item_review_time_index contains missing review_timestamp_ms values.")
if entity_review_daily_counts["facet_type"].astype(str).str.startswith("review_reputation", na=False).any():
    raise RuntimeError("Temporal entity counts must not use review-reputation facet types.")

expected_output_names = {
    ITEM_DOCS_PATH: "item_docs_herbal.parquet",
    ITEMS_FACETS_PATH: "items_facets_herbal.parquet",
    FACET_VOCAB_PATH: "facet_vocab_herbal.json",
    GRAPH_EDGES_PATH: "graph_edges_herbal.parquet",
    MANIFEST_PATH: "retrieval_artifact_manifest_herbal.json",
}
for path, expected_name in expected_output_names.items():
    if Path(path).name != expected_name:
        raise RuntimeError(f"Downstream filename changed: {path}")

historical_review_reputation_enabled_manifest = (
    int(len(review_item_facets)) > 0 and historical_review_covered_items > 0 and review_derived_item_text_rows > 0
)
review_reputation_graph_enabled_manifest = historical_review_graph_edges > 0
brand_in_functional_graph_manifest = False
brand_graph_enabled_manifest = brand_graph_edges_used > 0

manifest = {
    "stage": STAGE,
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "project_root": str(PROJECT_ROOT),
    "schema_path": str(SCHEMA_PATH),
    "output_paths": {
        **{name: str(path) for name, path in artifact_outputs.items()},
        "manifest": str(MANIFEST_PATH),
    },
    "evidence_scope": PRODUCTION_EVIDENCE_SCOPE,
    "historical_review_reputation_enabled": bool(historical_review_reputation_enabled_manifest),
    "historical_review_reputation_quarantined": False,
    "review_reputation_graph_enabled": bool(review_reputation_graph_enabled_manifest),
    "brand_in_functional_graph": bool(brand_in_functional_graph_manifest),
    "brand_graph_enabled": bool(brand_graph_enabled_manifest),
    "brand_in_retrieval_text": True,
    "brand_in_profile_source_text": True,
    "brand_in_synthetic_query": False,
    "raw_review_text_exported": bool(raw_review_text_exported),
    "dense_source": "dense_text",
    "sparse_source": "sparse_text",
    "item_facet_export_option": PRODUCTION_FACET_EXPORT_OPTION,
    "notebook_07_required_core_mask": CORE_GRAPH_MASK_DESCRIPTION,
    "notebook_07_required_brand_mask": BRAND_GRAPH_MASK_DESCRIPTION,
    "brand_graph_mask": BRAND_GRAPH_MASK_DESCRIPTION,
    "global_review_graph_mask": GLOBAL_REVIEW_GRAPH_MASK_DESCRIPTION,
    "item_docs_row_counts": {
        "item_docs_dense": int(len(item_docs_dense)),
        "item_docs_sparse": int(len(item_docs_sparse)),
        "item_docs": int(len(item_docs)),
    },
    "facet_row_counts": {
        "items_facets": int(len(item_facets)),
        "metadata_functional_facet_rows": int(len(core_item_facets)),
        "brand_facet_rows": int(len(brand_item_facets)),
        "review_reputation_facet_rows": int(len(review_item_facets)),
        "global_review_facet_rows": int(len(global_review_item_facets)),
        "by_facet_type": {str(key): int(value) for key, value in facet_counts.items()},
    },
    "graph_edge_counts": {
        "item_graph_long": int(len(graph_long)),
        "item_kg_edges": int(len(item_kg_edges)),
        "graph_edges": int(len(graph_edges)),
        "metadata_graph_edges": int(metadata_graph_edges),
        "brand_graph_edges": int(brand_graph_edges_used),
        "review_reputation_graph_edges": int(historical_review_graph_edges),
        "review_derived_rows_used": int(historical_review_graph_edges),
        "brand_rows_used": int(brand_graph_edges_used),
        "generic_anchor_rows_used": int(graph_edges["is_generic_category_anchor"].sum()),
        "generic_utility_rows_used": int(graph_edges["is_generic_utility_token"].sum()),
        "context_utility_rows_used": int(graph_edges["is_context_dependent_utility_token"].sum()),
        "disallowed_source_rows_used": int(graph_edges["is_disallowed_nonfacet_source"].sum()),
    },
    "temporal_artifacts": {
        "review_source_path": str(REVIEWS_PATH),
        "strict_pre_query_rule": TEMPORAL_REVIEW_STRICT_RULE,
        "temporal_windows_days": TEMPORAL_WINDOWS_DAYS,
        "temporal_facet_types": sorted(entity_review_daily_counts["facet_type"].dropna().astype(str).unique().tolist()),
        "entity_facet_source": "core_item_facets",
        "item_review_time_index_rows": int(len(item_review_time_index)),
        "item_review_daily_counts_rows": int(len(item_review_daily_counts)),
        "entity_review_daily_counts_rows": int(len(entity_review_daily_counts)),
        "item_temporal_summary_rows": int(len(item_temporal_summary)),
        "raw_review_text_loaded": False,
        "target_review_text_used": False,
    },
    "source_columns": {
        "dense_text_source": "canonical_text_dense",
        "sparse_text_source": "canonical_text_sparse",
        "profile_seed_source": "profile_source_text_dedup_seed",
        "profile_safe_facet_source": "profile_safe_facet_text",
        "brand_facet_source": "brand_facet_text",
        "query_safe_facet_source": "query_safe_facet_text",
        "functional_facet_source": "functional_facet_text",
    },
    "facet_specs": {
        "category_required_facet_cols": CATEGORY_REQUIRED_FACET_COLS,
        "category_optional_facet_specs": CATEGORY_OPTIONAL_FACET_SPECS,
        "metadata_specs_used": metadata_specs_used,
    },
    "policies": {
        "facet_policy_version": EXPECTED_FACET_POLICY_VERSION,
        "brand_policy": EXPECTED_BRAND_POLICY,
        "common_brand_policy": COMMON_BRAND_POLICY,
        "functional_facet_definition": COMMON_FUNCTIONAL_FACET_DEFINITION,
        "identifier_policy": EXPECTED_IDENTIFIER_POLICY,
        "identifier_diagnostic_excluded_from_retrieval_text": True,
        "raw_review_text_loaded": False,
        "review_timestamp_metadata_loaded": True,
        "target_review_text_used": False,
        "heldout_review_text_used": False,
        "rating_used": False,
        "sentiment_used": False,
        "llm_used": False,
        "user_artifacts_created": False,
    },
    "separated_text_validation": {
        "brand_in_functional_text_rows": int(brand_in_functional_text_rows),
        "possible_entity_fragment_rows": int(item_facets["is_possible_entity_fragment"].sum()),
    },
    "discontinued_removed_count": int(discontinued_removed_count),
    "created_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
with open(MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

print("Output:", MANIFEST_PATH)
print("Validation: Global Review retrieval artifacts passed")


Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/retrieval_artifact_manifest_herbal.json
Validation: Global Review retrieval artifacts passed


In [14]:
# ==== Export the Cross-Category Contract and Coverage Checks ====
common_retrieval_contract = dict(COMMON_RETRIEVAL_ARTIFACT_CONTRACT)
common_retrieval_contract["output_paths"] = {
    "common_retrieval_artifact_contract": str(COMMON_RETRIEVAL_ARTIFACT_CONTRACT_PATH),
    "facet_role_coverage": str(COMMON_FACET_ROLE_COVERAGE_PATH),
    "common_framework_flag_self_check": str(COMMON_FLAG_SELF_CHECK_PATH),
    "items_facets": str(ITEMS_FACETS_PATH),
    "item_docs": str(ITEM_DOCS_PATH),
    "facet_vocab": str(FACET_VOCAB_TABLE_PATH),
    "graph_edges": str(GRAPH_EDGES_PATH),
    "core_facet_filter_audit": str(CORE_FACET_FILTER_AUDIT_PATH),
}
common_retrieval_contract["item_facets_columns"] = list(item_facets.columns)
common_retrieval_contract["item_docs_columns"] = list(item_docs.columns)
common_retrieval_contract["production_text_fields"] = ["dense_text", "sparse_text"]
common_retrieval_contract["brand_graph_enabled"] = True
common_retrieval_contract["brand_in_retrieval_text"] = True
common_retrieval_contract["brand_in_profile_source_text"] = True
common_retrieval_contract["brand_in_synthetic_query"] = False
common_retrieval_contract["available_evidence_fields"] = [
    "historical_review_reputation_text",
    "review_reputation_facet_text",
    "review_reputation_only_text",
    "review_derived_signal_text",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
]
common_retrieval_contract["facet_flags"] = [
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_row_brand_value",
    "is_brand",
    "is_review_derived",
    "is_product_functional_facet",
    "is_query_safe",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]

common_framework_flag_self_check_df = pd.DataFrame([{
    "item_rows": int(len(item_docs)),
    "all_facet_rows": int(len(item_facets)),
    "core_facet_rows": int(len(core_item_facets)),
    "brand_facet_rows": int(len(brand_item_facets)),
    "review_facet_rows": int(len(review_item_facets)),
    "global_review_facet_rows": int(len(global_review_item_facets)),
    "core_dense_non_empty": int(item_docs["dense_text_core"].fillna("").str.strip().ne("").sum()),
    "core_sparse_non_empty": int(item_docs["sparse_text_core"].fillna("").str.strip().ne("").sum()),
    "brand_in_functional_facet_rows": int(
        (item_facets["is_brand"] & item_facets["is_product_functional_facet"]).sum()
    ),
    "review_rows_in_production_graph": int(graph_edges["is_review_derived"].sum()),
    "brand_rows_in_production_graph": int(graph_edges["is_brand_graph_facet"].sum()),
    "generic_rows_in_production_graph": int(
        (graph_edges["is_generic_category_anchor"] | graph_edges["is_generic_utility_token"]).sum()
    ),
    "status": "PASS",
}])

with open(COMMON_RETRIEVAL_ARTIFACT_CONTRACT_PATH, "w", encoding="utf-8") as file:
    json.dump(common_retrieval_contract, file, ensure_ascii=False, indent=2)
facet_role_coverage_df.to_csv(COMMON_FACET_ROLE_COVERAGE_PATH, index=False, encoding="utf-8-sig")
common_framework_flag_self_check_df.to_csv(COMMON_FLAG_SELF_CHECK_PATH, index=False, encoding="utf-8-sig")

print("Output:", COMMON_RETRIEVAL_ARTIFACT_CONTRACT_PATH)
print("Validation: common Global Review retrieval contract passed")


Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_retrieval_artifact_common_contract.json
Validation: common Global Review retrieval contract passed
